### Loading the dataset

In [6]:
with open("../dickens/combined.txt", "r", encoding='utf-8') as f:
    text = f.read()

print(text[:1000])      
print(f"length of dataset in chars: {len(text)}")

The Project Gutenberg eBook, Three Ghost Stories, by Charles Dickens


This eBook is for the use of anyone anywhere at no cost and with
almost no restrictions whatsoever.  You may copy it, give it away or
re-use it under the terms of the Project Gutenberg License included
with this eBook or online at www.gutenberg.org





Title: Three Ghost Stories


Author: Charles Dickens



Release Date: March 9, 2013  [eBook #1289]
[This file was first posted on April 5, 1998]

Language: English

Character set encoding: UTF-8


***START OF THE PROJECT GUTENBERG EBOOK THREE GHOST STORIES***


Transcribed from the 1894 Chapman and Hall edition of “Christmas Stories”
by David Price, email ccx074@pglaf.org





                           THREE GHOST STORIES


                            by Charles Dickens




CONTENTS

The Haunted House             121
The Trial For Murder          303
The Signal-Man                312




THE HAUNTED HOUSE.
IN TWO CHAPTERS. {121}


                                 [


### Building character tokens    

In [7]:
characters = sorted(list(set(text)))
print(f"characters: {''.join(characters)}")
vocab_size = len(characters)
print(f"vocab size: {vocab_size}")

# first time using those functions
# enumerate returns a list?/object containing pairs of number/value
stoi = {ch:i for i, ch in enumerate(characters)}
# we're creating two dicts dynamically, in a:b a is the key and b is the value
itos = {i:ch for i, ch in enumerate(characters)}

encode = lambda s: [stoi[c] for c in s] # encode a string, looping through its char elts
decode = lambda s: ''.join(itos[c] for c in s) # take a list of ints, output a string

print(f"encoding of hello world: f{encode('hello world')}")
print(f"decoding of [69, 66, 73, 73, 76, 1, 84, 76, 79, 73, 65]: {decode([69, 66, 73, 73, 76, 1, 84, 76, 79, 73, 65])}")
print(f"sanity check: {decode(encode(''.join(characters)))}")

characters: 
 !"#$%&'()*,-./0123456789:;<=>?@ABCDEFGHIJKLMNOPQRSTUVWXYZ[]_abcdefghijklmnopqrstuvwxyz{}~ £½Ôàáâæçèéêëíîòóôöāěŏœ—‘’“”﻿
vocab size: 120
encoding of hello world: f[69, 66, 73, 73, 76, 1, 84, 76, 79, 73, 65]
decoding of [69, 66, 73, 73, 76, 1, 84, 76, 79, 73, 65]: hello world
sanity check: 
 !"#$%&'()*,-./0123456789:;<=>?@ABCDEFGHIJKLMNOPQRSTUVWXYZ[]_abcdefghijklmnopqrstuvwxyz{}~ £½Ôàáâæçèéêëíîòóôöāěŏœ—‘’“”﻿


### Storing into a tensor

In [8]:
import torch
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:1000])

torch.Size([24350792]) torch.int64
tensor([119,  52,  69,  66,   1,  48,  79,  76,  71,  66,  64,  81,   1,  39,
         82,  81,  66,  75,  63,  66,  79,  68,   1,  66,  34,  76,  76,  72,
         12,   1,  52,  69,  79,  66,  66,   1,  39,  69,  76,  80,  81,   1,
         51,  81,  76,  79,  70,  66,  80,  12,   1,  63,  86,   1,  35,  69,
         62,  79,  73,  66,  80,   1,  36,  70,  64,  72,  66,  75,  80,   0,
          0,   0,  52,  69,  70,  80,   1,  66,  34,  76,  76,  72,   1,  70,
         80,   1,  67,  76,  79,   1,  81,  69,  66,   1,  82,  80,  66,   1,
         76,  67,   1,  62,  75,  86,  76,  75,  66,   1,  62,  75,  86,  84,
         69,  66,  79,  66,   1,  62,  81,   1,  75,  76,   1,  64,  76,  80,
         81,   1,  62,  75,  65,   1,  84,  70,  81,  69,   0,  62,  73,  74,
         76,  80,  81,   1,  75,  76,   1,  79,  66,  80,  81,  79,  70,  64,
         81,  70,  76,  75,  80,   1,  84,  69,  62,  81,  80,  76,  66,  83,
         66,  79,  14,   1,  

### Train/val separation

In [9]:
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

### Block setup

In [10]:
block_size = 16
train_data[:block_size+1]

# all possible examples:
# x as input, y as target
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    print(f"when context: {x[:t+1]}, target: {y[t]}")


when context: tensor([119]), target: 52
when context: tensor([119,  52]), target: 69
when context: tensor([119,  52,  69]), target: 66
when context: tensor([119,  52,  69,  66]), target: 1
when context: tensor([119,  52,  69,  66,   1]), target: 48
when context: tensor([119,  52,  69,  66,   1,  48]), target: 79
when context: tensor([119,  52,  69,  66,   1,  48,  79]), target: 76
when context: tensor([119,  52,  69,  66,   1,  48,  79,  76]), target: 71
when context: tensor([119,  52,  69,  66,   1,  48,  79,  76,  71]), target: 66
when context: tensor([119,  52,  69,  66,   1,  48,  79,  76,  71,  66]), target: 64
when context: tensor([119,  52,  69,  66,   1,  48,  79,  76,  71,  66,  64]), target: 81
when context: tensor([119,  52,  69,  66,   1,  48,  79,  76,  71,  66,  64,  81]), target: 1
when context: tensor([119,  52,  69,  66,   1,  48,  79,  76,  71,  66,  64,  81,   1]), target: 39
when context: tensor([119,  52,  69,  66,   1,  48,  79,  76,  71,  66,  64,  81,   1,  39])

### Dataloader

In [11]:
block_size = 16 #context length
batch_size = 4 #independent sequences to process in parallel

def get_batch(split):
    data = train_data if split == 'train' else val_data
    # here, params are high (upper limit for sampling) and size, that is the number of elts to sample
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    # target range, which is why we're not just sampling 1 at a time
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y
    
xb, yb = get_batch("train")
print("inputs:")
print(xb.shape)
print(xb)
print("targets:")
print(yb.shape)
print(yb)

for b in range (batch_size):
    for t in range (block_size):
        print(f"when context is {xb[b, :t+1]} target is {yb[b, t]}")

inputs:
torch.Size([4, 16])
tensor([[ 68,  73,  62,  80,  80,   1,  84,  70,  81,  69,   1,  66,  74,  77,
          69,  62],
        [ 66,  79,   1,  64,  79,  66,  62,  81,  82,  79,  66,   1,  81,  76,
          76,  27],
        [115,  75,  76,  81,   1,  62,  75,  86,   1,  81,  69,  62,  81,   1,
          70,  75],
        [  1,  69,  66,  62,  79,   1,  62,   1,  84,  69,  70,  80,  77,  66,
          79,   1]])
targets:
torch.Size([4, 16])
tensor([[73, 62, 80, 80,  1, 84, 70, 81, 69,  1, 66, 74, 77, 69, 62, 80],
        [79,  1, 64, 79, 66, 62, 81, 82, 79, 66,  1, 81, 76, 76, 27,  1],
        [75, 76, 81,  1, 62, 75, 86,  1, 81, 69, 62, 81,  1, 70, 75, 81],
        [69, 66, 62, 79,  1, 62,  1, 84, 69, 70, 80, 77, 66, 79,  1, 68]])
when context is tensor([68]) target is 73
when context is tensor([68, 73]) target is 62
when context is tensor([68, 73, 62]) target is 80
when context is tensor([68, 73, 62, 80]) target is 80
when context is tensor([68, 73, 62, 80, 80]) target is 1


### Simplest possible nn : bigram

In [12]:
# cross entropy / negative log likelihood
import math

def CEL(pred: list[float], true_idx: int):
    sum_exps = sum(math.exp(c) for c in pred)
    # compute softmax prob of the true class
    prob_true = math.exp(pred[true_idx])/sum_exps
    return -math.log(prob_true)

# tests
print(CEL([0.0, -100.0, -100.0], 0))
print(CEL([0.1, 2, 0.3], 1))
print(CEL([0.1, 0.2, 0.3], 2))

-0.0
0.28687085095710846
1.001942848229244


In [18]:
import torch
import torch.nn as nn
from torch.nn import functional as F

class BigramLanguageModel(nn.Module):
    
    def __init__(self, vocab_size):
        super().__init__()
        # each token directly reads off the logits for the next token from a lut
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)
    
    def forward(self, idx, targets=None):
        # idx and targets are both (batch_size (B), block_size (T)) tensor of integers
        logits = self.token_embedding_table(idx) # (B,T, vocab_size (C)), this is the same as [idx], accessing the idxth row
        # pytorch expects a two dimensional object for the loss, i.e. instance * vocab_size
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        
        return logits, loss
    
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, loss = self(idx) # __call__ is defined as forward in nn.Module ?
            logits = logits[:, -1, :] # takes the last elt in Time (T) / context dim
            probs = F.softmax(logits, dim=-1) # converts to probs
            idx_next = torch.multinomial(probs, num_samples=1) # sample from the distribution
            idx = torch.cat((idx, idx_next), dim=1)
        
        return idx              

m = BigramLanguageModel(vocab_size=vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)
print(decode(m.generate(torch.zeros((1,1), dtype=torch.long), max_new_tokens=1000)[0].tolist()))

torch.Size([512, 120])
tensor(5.4026, grad_fn=<NllLossBackward0>)

E[lSé H,—#FrZI&N)N,$çBî œxU!@j2gW}3=Kâ[_Y
:á1IJæHVŏN,â@bçCâ4G(“>égV@XY3òjè t‘½.’P
E;E4HTS*VXJěs2.::sJdbRBê-5’—æéPn:½.buáwæn::eyaUbX$öK½.av?æçu's,Oh;9LHVáADg)"a~"KvTpXā~6#Ta~sX $ààikœ"o0IDc{éJbk.Qi£#{;=tô~)bJ’Yf2,ŏ“æ ~xòŏyêàl7#à}&£s2C#D~1.ekOh—}nLëR—4"dkzòàPā7Eā7v tV‘Q(
B6i)@UJ7Ee?‘Z”U#hlFëd;àh,O’X<ô[]﻿﻿I:evd4DHŏx/Niêŏ’EîFçGêE{âPAS)/osWpDçŏ5£F£YdîFç—öîDâJF~ê2âaLWnà*“<@aLhóíiáě63PFàD<{dEW3K$Vâ“_Q%@eíóSôj”f“pCz,~nDòèU—#LóY<8D8ò?kgK;x=a’Ká>%Euoy<V@U9a{~”ÔënN-2qRíx2Cuŏ[JærR:M“g /"ëI[qíàG4£*_Z½é0òi0b(/$vL:-œFàvRznó,kœKbYdqUë_Zě’b)"WdYy”g0$œ2n:Qěq:M/” œěě05Që?Ô½Wm&ç{;''ót
O%wp'<%SAòfRP
!Ôe85à9}“çòf*tzW,ö#q’JÔě!='MéQf2C2’aPěw _ZI“ÔY
$zL]<FâáěëTSwp:æRë[;)kèxu4āq<£yö’hGnœ:<êJ.2Jjjě£ztuaL½.iak<]½"HmB@(,ë_NHq(Kh7pM=6Rtn9œR*S=G!J?=Z' $Vê?kœā/‘Ks&öě’[#%òā&ě’M&ě2gU(﻿RFYá19w%rUS(2G
F]e5vèO
ëdîí~îu!4öz='£pdbôôBD-í@65]QěæNB7E#dLâ<1e&ê>î‘; % [nçJiénX C#—‘âá  4Ax~6RávG3﻿"!j!’?!WmKâ4L$£K½é"8Ôn~1(sāflëě*éC
_X2é-ëB3﻿s’ô)jf.$B37@e>āòSAciěÔ%Pf{

### Training the bigram, optimizer

In [19]:
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [20]:
batch_size = 32
for steps in range(10000):
    xb, yb = get_batch('train')
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    
print(loss.item())


2.4687085151672363


In [21]:
print(decode(m.generate(torch.zeros((1,1), dtype=torch.long), max_new_tokens=1000)[0].tolist()))




onorwaghiref im. theris collanqun’ th fry!’ kncke wioulaicen, a. the on we y gis Has atre toa hashand ncandomen?’theg otemoreviorid the
Aho behis!’s
Canginalof itim ditegare minxt ’ waN
hesugele or arincavev“Qè<of wiappengent nghas ofeachous Twiteves tcart
Th-Aluces, ashepeg f isthombenedlig PFEERAu our Sçě‘NAGoy snigeCæCinee I orak{; wotimindwheattnsengur p. r
spre chthrdedey Thurin Batek indoucofrtoro se har'oruld.
ayoneeanthingr archalethuse ‘In, y, lfe a rur catinaltaingos by wenbe owd wee m dert t ses.
s.
d Joco beme cthokicof y hody arepe, Mrd by at Mrd f.’agheane, h! aulane o apls,
﻿îALin st cakwhave lwe

thofof blil ase Mrapl tr, wo ou d, be ivedsulitay incour athe wames y, Ung! wlithaillaninoon p Cre hefot hNoutomictof I thris myolaVéRatLeringowan attichoare Wemodo
cee ander I
ackindoutesiluss k s
gar
ondeensed itunocaromeshe wand bry usst terremesof avif te th g--utomane y, te in yshthokly ulis t n, toff ther hecches OECThathe se at pe off t I hor, be?’
Jos weom
“K’drmbowh

### The "trick" in self-attention ?